In [1]:
# Jetson Orin Nano setup
import sys, os
TTN_ROOT = '/home/ganeshnj-n1/tiny-ton'
sys.path.insert(0, TTN_ROOT + '/build/bindings')
sys.path.insert(0, TTN_ROOT + '/python')
os.environ['TTN_SM_VERSION'] = 'sm_87'  # Ampere sm_87
import _tiny_ton_core as core
import tiny_ton as tt
import numpy as np
print('has_cuda =', core.has_cuda())
print('SM = sm_87  (Ampere, Jetson Orin Nano)')


has_cuda = True
SM = sm_87  (Ampere, Jetson Orin Nano)


# tiny-ton on Jetson Orin Nano (sm_87)

**Everything is pre-built.** Just run Cell 1 (setup/imports) then run each section in order.

| Step | What it tests |
|------|---------------|
| Cell 1 | Setup: paths, sm_87, GPU check |
| Section 6 | CPU vs GPU helpers |
| Section 7 | vector_add (i32, f32, f16) |
| Section 8 | Math intrinsics (exp, log, sqrt...) |
| Section 9 | Reductions (reduce_sum, reduce_max) |
| Section 10 | ReLU |
| Section 11 | Gather / embedding lookup |
| Section 12 | Dot product & Matvec |
| Section 13 | Softmax |
| Section 14 | Fused attention (QKV) |
| Section 15 | Tiled GEMM |

In [2]:
# ── Test helpers ──────────────────────────────────────────────────────────
results = []

def compare(name, cpu, gpu, atol=1e-5):
    ok = np.allclose(cpu, gpu, atol=atol, rtol=1e-2)
    tag = 'PASS' if ok else 'FAIL'
    cpu_val = cpu.flat[0]
    gpu_val = gpu.flat[0]
    print(f'  {name:24s}  cpu={cpu_val:12.6f}  gpu={gpu_val:12.6f}  {tag}')
    if not ok:
        print(f"  {'':24s}  max_diff={np.max(np.abs(cpu.astype(np.float64) - gpu.astype(np.float64))):.6e}")
    results.append((name, ok))
    return ok

def compare_exact(name, cpu, gpu):
    ok = np.array_equal(cpu, gpu)
    tag = 'PASS' if ok else 'FAIL'
    print(f'  {name:24s}  cpu={cpu.flat[0]:12d}  gpu={gpu.flat[0]:12d}  {tag}')
    if not ok:
        mismatches = np.sum(cpu != gpu)
        print(f"  {'':24s}  {mismatches} mismatches out of {cpu.size}")
    results.append((name, ok))
    return ok

print('helpers loaded.')


helpers loaded.


## 5. Debug pipeline (stage-by-stage IR dump)

Shows every compilation stage: Python source -> AST -> TinyTon MLIR -> simulator assembly -> PTX.

## 6. CPU vs GPU — Setup

Import TinyTon and define a helper that compares a CPU (numpy) result against a GPU (TinyTon kernel) result.

## 7. CPU vs GPU — vector_add (i32, f32, f16)

For each dtype: compute `a + b` on the CPU with numpy, then run the same operation through the TinyTon GPU kernel, and compare.

In [3]:
@tt.jit
def vector_add(a_ptr, b_ptr, c_ptr, N):
    pid = tt.program_id(0)
    offsets = pid * 64 + tt.arange(0, 64)
    mask = offsets < N
    a = tt.load(a_ptr + offsets, mask=mask)
    b = tt.load(b_ptr + offsets, mask=mask)
    tt.store(c_ptr + offsets, a + b, mask=mask)

N = 256
grid = (N // 64,)
np.random.seed(0)

for dtype_name, dtype, atol in [('i32', np.int32, 0),
                                 ('f32', np.float32, 1e-6),
                                 ('f16', np.float16, 1e-2)]:
    print(f'\n--- vector_add {dtype_name} ---')
    if dtype == np.int32:
        a = np.arange(N, dtype=dtype)
        b = np.arange(N, dtype=dtype)
    else:
        a = np.random.randn(N).astype(dtype)
        b = np.random.randn(N).astype(dtype)

    cpu_c = (a + b).astype(dtype)

    gpu_c = np.zeros(N, dtype=dtype)
    vector_add[grid](a.copy(), b.copy(), gpu_c, N)

    if dtype == np.int32:
        compare_exact(f'add_{dtype_name}', cpu_c, gpu_c)
    else:
        compare(f'add_{dtype_name}', cpu_c, gpu_c, atol=atol)


--- vector_add i32 ---
  add_i32                   cpu=           0  gpu=           0  PASS

--- vector_add f32 ---
  add_f32                   cpu=    1.038455  gpu=    1.038455  PASS

--- vector_add f16 ---
  add_f16                   cpu=    0.517578  gpu=    0.517578  PASS


## 8. CPU vs GPU — Math intrinsics (exp, log, sqrt, rsqrt, abs, max)

For each intrinsic: compute on CPU with numpy, then run through the TinyTon GPU kernel, and compare.

In [4]:
@tt.jit
def kern_exp(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.exp(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_log(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.log(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_sqrt(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.sqrt(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_rsqrt(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.rsqrt(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_abs(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.abs(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_max(a_ptr, b_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    a = tt.load(a_ptr + off, mask=mask)
    b = tt.load(b_ptr + off, mask=mask)
    tt.store(dst + off, tt.max(a, b), mask=mask)

N = 256
grid = (N // 64,)
np.random.seed(42)

for dtype_name, dtype, atol in [('f32', np.float32, 1e-5), ('f16', np.float16, 5e-2)]:
    print(f'\n--- math intrinsics {dtype_name} ---')
    pos = (np.abs(np.random.randn(N)) + 0.01).astype(dtype)
    mixed = (np.random.randn(N) * 2).astype(dtype)

    for name, kernel, np_fn, inp in [
        ('exp',  kern_exp,  np.exp,  mixed),
        ('log',  kern_log,  np.log,  pos),
        ('sqrt', kern_sqrt, np.sqrt, pos),
        ('abs',  kern_abs,  np.abs,  mixed),
    ]:
        cpu_out = np_fn(inp).astype(dtype)
        gpu_out = np.zeros(N, dtype=dtype)
        kernel[grid](inp.copy(), gpu_out, N)
        compare(f'{name}_{dtype_name}', cpu_out, gpu_out, atol=atol)

    cpu_rsqrt = (1.0 / np.sqrt(pos.astype(np.float32))).astype(dtype)
    gpu_rsqrt = np.zeros(N, dtype=dtype)
    kern_rsqrt[grid](pos.copy(), gpu_rsqrt, N)
    compare(f'rsqrt_{dtype_name}', cpu_rsqrt, gpu_rsqrt, atol=atol)

    a = (np.random.randn(N) * 5).astype(dtype)
    b = (np.random.randn(N) * 5).astype(dtype)
    cpu_max = np.maximum(a, b)
    gpu_max = np.zeros(N, dtype=dtype)
    kern_max[grid](a.copy(), b.copy(), gpu_max, N)
    compare(f'max_{dtype_name}', cpu_max, gpu_max, atol=atol)


--- math intrinsics f32 ---
  exp_f32                   cpu=   12.601582  gpu=   12.601582  PASS
  log_f32                   cpu=   -0.679808  gpu=   -0.679808  PASS
  sqrt_f32                  cpu=    0.711839  gpu=    0.711839  PASS
  abs_f32                   cpu=    2.533822  gpu=    2.533822  PASS
  rsqrt_f32                 cpu=    1.404813  gpu=    1.404813  PASS
  max_f32                   cpu=   -1.194740  gpu=   -1.194740  PASS

--- math intrinsics f16 ---
  exp_f16                   cpu=    1.306641  gpu=    1.306641  PASS
  log_f16                   cpu=    0.552246  gpu=    0.552246  PASS
  sqrt_f16                  cpu=    1.318359  gpu=    1.318359  PASS
  abs_f16                   cpu=    0.267090  gpu=    0.267090  PASS
  rsqrt_f16                 cpu=    0.758789  gpu=    0.758789  PASS
  max_f16                   cpu=    2.568359  gpu=    2.568359  PASS


## 9. CPU vs GPU — Reductions (reduce_sum, reduce_max)

For each block: compute the reduction on CPU with numpy, then run through the TinyTon GPU kernel (warp-shuffle + shared memory), and compare.

In [5]:
@tt.jit
def kern_reduce_sum(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    total = tt.reduce_sum(x)
    tt.store(dst + pid, total)

@tt.jit
def kern_reduce_max(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    mx = tt.reduce_max(x)
    tt.store(dst + pid, mx)

N = 256
BLOCK = 64
grid = (N // BLOCK,)
np.random.seed(123)

for dtype_name, dtype, atol in [('f32', np.float32, 1e-4)]:
    print(f'\n--- reductions {dtype_name} ---')
    x = np.random.randn(N).astype(dtype)

    cpu_sums = np.array([np.sum(x[i*BLOCK:(i+1)*BLOCK]) for i in range(N // BLOCK)], dtype=dtype)
    gpu_sums = np.zeros(N // BLOCK, dtype=dtype)
    kern_reduce_sum[grid](x.copy(), gpu_sums, N)
    for i in range(N // BLOCK):
        compare(f'reduce_sum_blk{i}_{dtype_name}',
                cpu_sums[i:i+1], gpu_sums[i:i+1], atol=atol)

    cpu_maxs = np.array([np.max(x[i*BLOCK:(i+1)*BLOCK]) for i in range(N // BLOCK)], dtype=dtype)
    gpu_maxs = np.zeros(N // BLOCK, dtype=dtype)
    kern_reduce_max[grid](x.copy(), gpu_maxs, N)
    for i in range(N // BLOCK):
        compare(f'reduce_max_blk{i}_{dtype_name}',
                cpu_maxs[i:i+1], gpu_maxs[i:i+1], atol=atol)


--- reductions f32 ---
  reduce_sum_blk0_f32       cpu=    4.596138  gpu=    4.596138  PASS
  reduce_sum_blk1_f32       cpu=    0.032585  gpu=    0.032585  PASS
  reduce_sum_blk2_f32       cpu=    0.028552  gpu=    0.028553  PASS
  reduce_sum_blk3_f32       cpu=   -8.825704  gpu=   -8.825704  PASS
  reduce_max_blk0_f32       cpu=    2.392365  gpu=    2.392365  PASS
  reduce_max_blk1_f32       cpu=    2.598304  gpu=    2.598304  PASS
  reduce_max_blk2_f32       cpu=    1.805970  gpu=    1.805970  PASS
  reduce_max_blk3_f32       cpu=    2.200702  gpu=    2.200702  PASS


## 10. CPU vs GPU — ReLU (Rectified Linear Unit)

Element-wise `max(x, 0)`. Composed from existing `tt.max` — no new C++ ops.

In [6]:
@tt.jit
def kern_relu(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    y = tt.relu(x)
    tt.store(dst + off, y, mask=mask)

N = 256
grid = (N // 64,)
np.random.seed(99)

for dtype_name, dtype, atol in [('f32', np.float32, 0.0), ('f16', np.float16, 0.0)]:
    print(f'\n--- relu {dtype_name} ---')
    x = (np.random.randn(N) * 5).astype(dtype)
    cpu_relu = np.maximum(x, dtype(0))
    gpu_relu = np.zeros(N, dtype=dtype)
    kern_relu[grid](x.copy(), gpu_relu, N)
    compare(f'relu_{dtype_name}', cpu_relu, gpu_relu, atol=atol)


--- relu f32 ---
  relu_f32                  cpu=    0.000000  gpu=    0.000000  PASS

--- relu f16 ---
  relu_f16                  cpu=    0.000000  gpu=    0.000000  PASS


## 11. CPU vs GPU — Gather (Embedding Lookup)

Index a row from a 2D table stored in flat memory — the core of embedding lookups.
Pure address arithmetic on top of existing `tt.load` — no new C++ ops.

In [7]:
@tt.jit
def gather_kernel(table_ptr, index, dst_ptr, n_embd):
    tid = tt.arange(0, 64)
    mask = tid < n_embd
    val = tt.load(table_ptr + index * n_embd + tid, mask=mask)
    tt.store(dst_ptr + tid, val, mask=mask)

np.random.seed(7)
rows, cols = 8, 16
table = np.random.randn(rows, cols).astype(np.float32)
table_flat = table.flatten()

print('\n--- gather f32 ---')
for idx in [0, 3, 7]:
    expected = table[idx]
    dst = np.zeros(cols, dtype=np.float32)
    gather_kernel[(1,)](table_flat.copy(), idx, dst, cols)
    compare(f'gather_row{idx}_f32', expected, dst, atol=0.0)


--- gather f32 ---
  gather_row0_f32           cpu=    1.690526  gpu=    1.690526  PASS
  gather_row3_f32           cpu=    0.269412  gpu=    0.269412  PASS
  gather_row7_f32           cpu=    1.055948  gpu=    1.055948  PASS


## 12. CPU vs GPU — Dot Product & Matvec

Dot product via `tt.reduce_sum(a * b)`. Matvec: one dot per block, `grid = (out_features,)`.
Composed from existing `tt.mul` + `tt.reduce_sum` — no new C++ ops.

In [8]:
@tt.jit
def dot_kernel(a_ptr, b_ptr, dst_ptr, N):
    tid = tt.arange(0, 64)
    mask = tid < N
    a = tt.load(a_ptr + tid, mask=mask)
    b = tt.load(b_ptr + tid, mask=mask)
    result = tt.reduce_sum(a * b)
    tt.store(dst_ptr, result)

@tt.jit
def matvec_kernel(W_ptr, x_ptr, y_ptr, in_features):
    pid = tt.program_id(0)
    tid = tt.arange(0, 64)
    mask = tid < in_features
    w = tt.load(W_ptr + pid * in_features + tid, mask=mask)
    x = tt.load(x_ptr + tid, mask=mask)
    dot = tt.reduce_sum(w * x)
    tt.store(y_ptr + pid, dot)

np.random.seed(55)

# --- dot product ---
print('\n--- dot f32 ---')
N_dot = 16
a = np.random.randn(N_dot).astype(np.float32)
b = np.random.randn(N_dot).astype(np.float32)
expected_dot = np.array([np.dot(a, b)], dtype=np.float32)
gpu_dot = np.zeros(1, dtype=np.float32)
dot_kernel[(1,)](a.copy(), b.copy(), gpu_dot, N_dot)
compare('dot_f32', expected_dot, gpu_dot, atol=1e-4)

# --- matvec ---
print('\n--- matvec f32 ---')
out_f, in_f = 4, 16
W = np.random.randn(out_f, in_f).astype(np.float32)
x = np.random.randn(in_f).astype(np.float32)
expected_y = (W @ x).astype(np.float32)
W_flat = W.flatten()
gpu_y = np.zeros(out_f, dtype=np.float32)
matvec_kernel[(out_f,)](W_flat.copy(), x.copy(), gpu_y, in_f)
for i in range(out_f):
    compare(f'matvec_row{i}_f32', expected_y[i:i+1], gpu_y[i:i+1], atol=1e-4)


--- dot f32 ---
  dot_f32                   cpu=   -4.910771  gpu=   -4.910771  PASS

--- matvec f32 ---
  matvec_row0_f32           cpu=    0.538036  gpu=    0.538035  PASS
  matvec_row1_f32           cpu=    0.680969  gpu=    0.680969  PASS
  matvec_row2_f32           cpu=    0.018574  gpu=    0.018574  PASS
  matvec_row3_f32           cpu=   -2.092643  gpu=   -2.092644  PASS


## 13. CPU vs GPU — Softmax (5 kernel launches)

Numerically stable softmax: `reduce_max` -> `sub` -> `exp` -> `reduce_sum` -> `div`.
Each step is a separate kernel launch — no fusion. All ops already exist.

In [9]:
@tt.jit
def sm_reduce_max(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    mx = tt.reduce_max(x)
    tt.store(dst + pid, mx)

@tt.jit
def sm_sub_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x - s, mask=mask)

@tt.jit
def sm_exp(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    tt.store(dst + off, tt.exp(x), mask=mask)

@tt.jit
def sm_reduce_sum(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    total = tt.reduce_sum(x)
    tt.store(dst + pid, total)

@tt.jit
def sm_div_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x / s, mask=mask)

def gpu_softmax(x, out, N):
    grid = (max(1, (N + 63) // 64),)
    tmp_max = np.zeros(1, dtype=x.dtype)
    tmp_exp = np.zeros(N, dtype=x.dtype)
    tmp_sum = np.zeros(1, dtype=x.dtype)
    sm_reduce_max[(1,)](x, tmp_max, N)
    sm_sub_scalar[grid](x, tmp_max, tmp_exp, N)
    sm_exp[grid](tmp_exp, tmp_exp, N)
    sm_reduce_sum[(1,)](tmp_exp, tmp_sum, N)
    sm_div_scalar[grid](tmp_exp, tmp_sum, out, N)

np.random.seed(77)
print('\n--- softmax f32 ---')
for N_sm in [16, 32]:
    x = np.random.randn(N_sm).astype(np.float32)
    shifted = x - np.max(x)
    cpu_sm = np.exp(shifted) / np.sum(np.exp(shifted))
    gpu_sm = np.zeros(N_sm, dtype=np.float32)
    gpu_softmax(x.copy(), gpu_sm, N_sm)
    compare(f'softmax_N{N_sm}_f32', cpu_sm, gpu_sm, atol=1e-5)
    print(f'  sum={np.sum(gpu_sm):.6f}')


--- softmax f32 ---
  softmax_N16_f32           cpu=    0.049928  gpu=    0.049928  PASS
  sum=1.000000
  softmax_N32_f32           cpu=    0.008474  gpu=    0.008474  PASS
  sum=1.000000


## 14. CPU vs GPU — RMSNorm (4 kernel launches)

Root Mean Square Layer Normalization: `x * rsqrt(mean(x^2) + eps)`.
Composed from `square` -> `reduce_sum` -> `rsqrt_mean` -> `mul_scalar`. No new C++ ops.

In [10]:
@tt.jit
def rn_square(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    tt.store(dst + off, x * x, mask=mask)

@tt.jit
def rn_reduce_sum(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    total = tt.reduce_sum(x)
    tt.store(dst + pid, total)

@tt.jit
def rn_rsqrt_mean(sum_ptr, n_ptr, out_ptr):
    tid = tt.arange(0, 64)
    s = tt.load(sum_ptr)
    n = tt.load(n_ptr)
    mean_eps = s / n + 1e-5
    scale = tt.rsqrt(mean_eps)
    tt.store(out_ptr, scale)

@tt.jit
def rn_mul_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x * s, mask=mask)

def gpu_rmsnorm(x, out, N):
    grid = (max(1, (N + 63) // 64),)
    tmp_sq  = np.zeros(N, dtype=x.dtype)
    tmp_sum = np.zeros(1, dtype=x.dtype)
    tmp_scl = np.zeros(1, dtype=x.dtype)
    n_arr   = np.array([float(N)], dtype=x.dtype)
    rn_square[grid](x, tmp_sq, N)
    rn_reduce_sum[(1,)](tmp_sq, tmp_sum, N)
    rn_rsqrt_mean[(1,)](tmp_sum, n_arr, tmp_scl)
    rn_mul_scalar[grid](x, tmp_scl, out, N)

np.random.seed(88)
print('\n--- rmsnorm f32 ---')
for N_rn in [16, 32]:
    x = np.random.randn(N_rn).astype(np.float32)
    rms = np.sqrt(np.mean(x ** 2) + 1e-5)
    cpu_rn = (x / rms).astype(np.float32)
    gpu_rn = np.zeros(N_rn, dtype=np.float32)
    gpu_rmsnorm(x.copy(), gpu_rn, N_rn)
    compare(f'rmsnorm_N{N_rn}_f32', cpu_rn, gpu_rn, atol=1e-5)


--- rmsnorm f32 ---
  rmsnorm_N16_f32           cpu=    0.105800  gpu=    0.105800  PASS
  rmsnorm_N32_f32           cpu=   -0.779369  gpu=   -0.779369  PASS


## 15. CPU vs GPU — Linear (matvec, one output per block)

`y = W @ x` using `matvec_kernel` — each block computes one dot product.
Also tests chained layers: `x -> W1 -> relu -> W2 -> y` (MLP-like pipeline).

In [11]:
@tt.jit
def linear_kernel(W_ptr, x_ptr, y_ptr, in_features):
    pid = tt.program_id(0)
    tid = tt.arange(0, 64)
    mask = tid < in_features
    w = tt.load(W_ptr + pid * in_features + tid, mask=mask)
    x = tt.load(x_ptr + tid, mask=mask)
    dot = tt.reduce_sum(w * x)
    tt.store(y_ptr + pid, dot)

@tt.jit
def lin_relu(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    y = tt.relu(x)
    tt.store(dst + off, y, mask=mask)

def linear(W_flat, x, y, out_features, in_features):
    linear_kernel[(out_features,)](W_flat, x, y, in_features)

np.random.seed(33)

# --- single linear (16 -> 8, attention-projection-like) ---
print('\n--- linear f32 ---')
in_f, out_f = 16, 8
W = np.random.randn(out_f, in_f).astype(np.float32)
x = np.random.randn(in_f).astype(np.float32)
cpu_y = (W @ x).astype(np.float32)
gpu_y = np.zeros(out_f, dtype=np.float32)
linear(W.flatten().copy(), x.copy(), gpu_y, out_f, in_f)
compare('linear_16to8_f32', cpu_y, gpu_y, atol=1e-4)

# --- chained: x -> W1 -> relu -> W2 -> y (MLP-like) ---
print('\n--- chained linear (MLP) f32 ---')
in_f, hid, out_f = 16, 32, 16
W1 = np.random.randn(hid, in_f).astype(np.float32)
W2 = np.random.randn(out_f, hid).astype(np.float32)
x = np.random.randn(in_f).astype(np.float32)
cpu_h = np.maximum(W1 @ x, 0.0).astype(np.float32)
cpu_out = (W2 @ cpu_h).astype(np.float32)

gpu_h = np.zeros(hid, dtype=np.float32)
gpu_h_relu = np.zeros(hid, dtype=np.float32)
gpu_out = np.zeros(out_f, dtype=np.float32)
linear(W1.flatten().copy(), x.copy(), gpu_h, hid, in_f)
lin_relu[(1,)](gpu_h, gpu_h_relu, hid)
linear(W2.flatten().copy(), gpu_h_relu, gpu_out, out_f, hid)
compare('chained_mlp_f32', cpu_out, gpu_out, atol=1e-3)


--- linear f32 ---
  linear_16to8_f32          cpu=    5.310970  gpu=    5.310971  PASS

--- chained linear (MLP) f32 ---
  chained_mlp_f32           cpu=   12.304980  gpu=   12.304980  PASS


True

## 16. CPU vs GPU — Cross-Entropy Loss (7 kernel launches)

`loss = -log(softmax(logits)[target])`. Composes softmax (5 launches) + scalar gather + negative log.
No new C++ ops — reuses existing softmax, `tt.load` for gather, and `tt.log`.

In [12]:
@tt.jit
def ce_reduce_max(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    mx = tt.reduce_max(x)
    tt.store(dst + pid, mx)

@tt.jit
def ce_sub_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x - s, mask=mask)

@tt.jit
def ce_exp(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    tt.store(dst + off, tt.exp(x), mask=mask)

@tt.jit
def ce_reduce_sum(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    total = tt.reduce_sum(x)
    tt.store(dst + pid, total)

@tt.jit
def ce_div_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x / s, mask=mask)

@tt.jit
def ce_gather_scalar(src, index, dst):
    tid = tt.arange(0, 64)
    val = tt.load(src + index)
    tt.store(dst, val)

@tt.jit
def ce_neg_log(src, dst):
    tid = tt.arange(0, 64)
    val = tt.load(src)
    result = 0.0 - tt.log(val)
    tt.store(dst, result)

def gpu_softmax_ce(x, out, N):
    grid = (max(1, (N + 63) // 64),)
    tmp_max = np.zeros(1, dtype=x.dtype)
    tmp_exp = np.zeros(N, dtype=x.dtype)
    tmp_sum = np.zeros(1, dtype=x.dtype)
    ce_reduce_max[(1,)](x, tmp_max, N)
    ce_sub_scalar[grid](x, tmp_max, tmp_exp, N)
    ce_exp[grid](tmp_exp, tmp_exp, N)
    ce_reduce_sum[(1,)](tmp_exp, tmp_sum, N)
    ce_div_scalar[grid](tmp_exp, tmp_sum, out, N)

def gpu_cross_entropy(logits, target, N):
    probs = np.zeros(N, dtype=logits.dtype)
    gpu_softmax_ce(logits, probs, N)
    p = np.zeros(1, dtype=logits.dtype)
    ce_gather_scalar[(1,)](probs, target, p)
    loss = np.zeros(1, dtype=logits.dtype)
    ce_neg_log[(1,)](p, loss)
    return loss[0]

np.random.seed(44)
N_ce = 16
logits = (np.random.randn(N_ce) * 2).astype(np.float32)

print('\n--- cross_entropy f32 ---')
for tgt in [0, 7, 15]:
    shifted = logits - np.max(logits)
    cpu_loss = -(shifted[tgt] - np.log(np.sum(np.exp(shifted))))
    gpu_loss = gpu_cross_entropy(logits.copy(), tgt, N_ce)
    cpu_arr = np.array([cpu_loss], dtype=np.float32)
    gpu_arr = np.array([gpu_loss], dtype=np.float32)
    compare(f'cross_entropy_tgt{tgt}_f32', cpu_arr, gpu_arr, atol=1e-4)


--- cross_entropy f32 ---
  cross_entropy_tgt0_f32    cpu=    5.872480  gpu=    5.872480  PASS
  cross_entropy_tgt7_f32    cpu=    4.196074  gpu=    4.196074  PASS
  cross_entropy_tgt15_f32   cpu=    3.649539  gpu=    3.649538  PASS


## 17. CPU vs GPU — Attention (12 kernel launches)

Single-head scaled dot-product attention: Q/K/V projections (3 launches) + scores via matvec (1) + scale (1) + softmax (5) + weighted sum via matvec (1) + output projection (1) = 12 total. Reuses all existing kernels.

In [13]:
@tt.jit
def attn_linear(W_ptr, x_ptr, y_ptr, in_features):
    pid = tt.program_id(0)
    tid = tt.arange(0, 64)
    mask = tid < in_features
    w = tt.load(W_ptr + pid * in_features + tid, mask=mask)
    x = tt.load(x_ptr + tid, mask=mask)
    dot = tt.reduce_sum(w * x)
    tt.store(y_ptr + pid, dot)

@tt.jit
def attn_reduce_max(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    mx = tt.reduce_max(x)
    tt.store(dst + pid, mx)

@tt.jit
def attn_sub_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x - s, mask=mask)

@tt.jit
def attn_exp(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    tt.store(dst + off, tt.exp(x), mask=mask)

@tt.jit
def attn_reduce_sum(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    total = tt.reduce_sum(x)
    tt.store(dst + pid, total)

@tt.jit
def attn_div_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x / s, mask=mask)

def gpu_softmax_attn(x, out, N):
    grid = (max(1, (N + 63) // 64),)
    tmp_max = np.zeros(1, dtype=np.float32)
    tmp_exp = np.zeros(N, dtype=np.float32)
    tmp_sum = np.zeros(1, dtype=np.float32)
    attn_reduce_max[(1,)](x, tmp_max, N)
    attn_sub_scalar[grid](x, tmp_max, tmp_exp, N)
    attn_exp[grid](tmp_exp, tmp_exp, N)
    attn_reduce_sum[(1,)](tmp_exp, tmp_sum, N)
    attn_div_scalar[grid](tmp_exp, tmp_sum, out, N)

def gpu_attention(x, Wq, Wk, Wv, Wo, K_cache, V_cache, n_embd):
    q = np.zeros(n_embd, dtype=np.float32)
    k = np.zeros(n_embd, dtype=np.float32)
    v = np.zeros(n_embd, dtype=np.float32)
    attn_linear[(n_embd,)](Wq, x, q, n_embd)
    attn_linear[(n_embd,)](Wk, x, k, n_embd)
    attn_linear[(n_embd,)](Wv, x, v, n_embd)
    K_cache.append(k.copy())
    V_cache.append(v.copy())
    K = np.ascontiguousarray(np.vstack(K_cache))
    V = np.vstack(V_cache)
    seq_len = len(K_cache)
    scores = np.zeros(seq_len, dtype=np.float32)
    attn_linear[(seq_len,)](K.flatten(), q, scores, n_embd)
    sqrt_d = np.array([np.sqrt(float(n_embd))], dtype=np.float32)
    scores_scaled = np.zeros(seq_len, dtype=np.float32)
    attn_div_scalar[(1,)](scores, sqrt_d, scores_scaled, seq_len)
    weights = np.zeros(seq_len, dtype=np.float32)
    gpu_softmax_attn(scores_scaled, weights, seq_len)
    V_T = np.ascontiguousarray(V.T)
    attn_out = np.zeros(n_embd, dtype=np.float32)
    attn_linear[(n_embd,)](V_T.flatten(), weights, attn_out, seq_len)
    output = np.zeros(n_embd, dtype=np.float32)
    attn_linear[(n_embd,)](Wo, attn_out, output, n_embd)
    return output

np.random.seed(42)
n_embd = 16
n_tokens = 4
Wq = (np.random.randn(n_embd, n_embd) * 0.1).astype(np.float32)
Wk = (np.random.randn(n_embd, n_embd) * 0.1).astype(np.float32)
Wv = (np.random.randn(n_embd, n_embd) * 0.1).astype(np.float32)
Wo = (np.random.randn(n_embd, n_embd) * 0.1).astype(np.float32)
tokens = [np.random.randn(n_embd).astype(np.float32) for _ in range(n_tokens)]

print('\n--- attention f32 (12 launches per position) ---')
K_gpu, V_gpu = [], []
K_ref, V_ref = [], []
for t in range(n_tokens):
    x = tokens[t]
    gpu_out = gpu_attention(x.copy(),
                            Wq.flatten().copy(), Wk.flatten().copy(),
                            Wv.flatten().copy(), Wo.flatten().copy(),
                            K_gpu, V_gpu, n_embd)
    q_np = Wq @ x; k_np = Wk @ x; v_np = Wv @ x
    K_ref.append(k_np.copy()); V_ref.append(v_np.copy())
    K_np = np.vstack(K_ref); V_np = np.vstack(V_ref)
    scores_np = K_np @ q_np / np.sqrt(float(n_embd))
    shifted = scores_np - np.max(scores_np)
    w_np = np.exp(shifted) / np.sum(np.exp(shifted))
    ref_out = Wo @ (V_np.T @ w_np)
    compare(f'attention_pos{t}_f32', gpu_out, ref_out, atol=1e-3)


--- attention f32 (12 launches per position) ---
  attention_pos0_f32        cpu=   -0.055581  gpu=   -0.055581  PASS
  attention_pos1_f32        cpu=   -0.042953  gpu=   -0.042953  PASS
  attention_pos2_f32        cpu=    0.030160  gpu=    0.030160  PASS
  attention_pos3_f32        cpu=    0.073833  gpu=    0.073833  PASS


## 18. Fused softmax — 5 launches → 1

Collapse reduce_max + sub + exp + reduce_sum + div into a single kernel. All intermediate values stay in thread registers — no global memory writes between steps.

**Masked thread correction**: threads with `tid >= N` load 0.0 and contribute `exp(-max)` to the raw reduce_sum. We pass `float(64 - N)` as a scalar and subtract their contribution: `s_corrected = s_raw - n_masked * exp(-mx)`.

In [14]:
import numpy as np
import time

# --- fused softmax: 1 kernel launch -----------------------------------------

@tt.jit
def fused_softmax_kernel(src, dst, N, n_masked_ptr):
    tid  = tt.arange(0, 64)
    mask = tid < N
    x    = tt.load(src + tid, mask=mask)
    mx   = tt.reduce_max(x)
    e    = tt.exp(x - mx)
    s    = tt.reduce_sum(e)
    nm   = tt.load(n_masked_ptr)
    s    = s - nm * tt.exp(0.0 - mx)
    tt.store(dst + tid, e / s, mask=mask)

def gpu_fused_softmax(x, out, N):
    n_masked = np.array([float(64 - N)], dtype=np.float32)
    fused_softmax_kernel[(1,)](x, out, N, n_masked)

# --- 5-kernel softmax (Stage 1 baseline) -------------------------------------

@tt.jit
def kern_reduce_max(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    mx = tt.reduce_max(x)
    tt.store(dst + pid, mx)

@tt.jit
def kern_sub_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x - s, mask=mask)

@tt.jit
def kern_exp(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    tt.store(dst + off, tt.exp(x), mask=mask)

@tt.jit
def kern_reduce_sum(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    total = tt.reduce_sum(x)
    tt.store(dst + pid, total)

@tt.jit
def kern_div_scalar(src, scalar_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    s = tt.load(scalar_ptr)
    tt.store(dst + off, x / s, mask=mask)

def gpu_5k_softmax(x, out, N):
    grid = (max(1, (N + 63) // 64),)
    tmp_max = np.zeros(1, dtype=x.dtype)
    tmp_exp = np.zeros(N, dtype=x.dtype)
    tmp_sum = np.zeros(1, dtype=x.dtype)
    kern_reduce_max[(1,)](x, tmp_max, N)
    kern_sub_scalar[grid](x, tmp_max, tmp_exp, N)
    kern_exp[grid](tmp_exp, tmp_exp, N)
    kern_reduce_sum[(1,)](tmp_exp, tmp_sum, N)
    kern_div_scalar[grid](tmp_exp, tmp_sum, out, N)

# --- correctness check -------------------------------------------------------

def numpy_softmax(x):
    shifted = x - np.max(x)
    exps = np.exp(shifted)
    return exps / np.sum(exps)

print("=== Correctness ===")
all_ok = True
for N in [4, 16, 27]:
    x = np.random.randn(N).astype(np.float32)
    expected = numpy_softmax(x)

    out_fused = np.zeros(N, dtype=np.float32)
    gpu_fused_softmax(x.copy(), out_fused, N)

    out_5k = np.zeros(N, dtype=np.float32)
    gpu_5k_softmax(x.copy(), out_5k, N)

    ok = (np.allclose(out_fused, expected, atol=1e-5) and
          np.allclose(out_fused, out_5k, atol=1e-5) and
          np.allclose(np.sum(out_fused), 1.0, atol=1e-5))
    print(f"  N={N:2d}: {'PASS' if ok else 'FAIL'}  sum={np.sum(out_fused):.6f}")
    all_ok = all_ok and ok

assert all_ok, "correctness check failed"

# --- timing comparison -------------------------------------------------------

REPS = 500
for N, label in [(27, "vocab logits (N=27)"), (16, "attention scores (N=16)")]:
    x    = np.random.randn(N).astype(np.float32)
    out1 = np.zeros(N, dtype=np.float32)
    out2 = np.zeros(N, dtype=np.float32)

    # warm-up
    for _ in range(5):
        gpu_fused_softmax(x, out1, N)
        gpu_5k_softmax(x, out2, N)

    t0 = time.perf_counter()
    for _ in range(REPS):
        gpu_fused_softmax(x, out1, N)
    t_fused = (time.perf_counter() - t0) / REPS * 1e6

    t0 = time.perf_counter()
    for _ in range(REPS):
        gpu_5k_softmax(x, out2, N)
    t_5k = (time.perf_counter() - t0) / REPS * 1e6

    print(f"\n{label}:")
    print(f"  5-kernel softmax: {t_5k:.1f} µs/call  (5 launches)")
    print(f"  fused  softmax:   {t_fused:.1f} µs/call  (1 launch)")
    print(f"  speedup:          {t_5k / t_fused:.2f}x")

=== Correctness ===
  N= 4: FAIL  sum=nan
  N=16: FAIL  sum=nan
  N=27: FAIL  sum=nan


AssertionError: correctness check failed

## 20. Constexpr block size — tt.constexpr

Today every kernel uses `tt.arange(0, 64)` — 64 threads per block regardless of
input size.  At `n_embd=16` that means 75% of threads are idle (masked out).

`tt.constexpr` lets you set the block size at the call site.  The value is baked
into the compiled PTX; different values produce separate cache entries.

```python
@tt.jit
def fused_softmax_constexpr(src, dst, N, BLOCK: tt.constexpr):
    tid  = tt.arange(0, BLOCK)   # ← block size from call site
    ...

fused_softmax_constexpr[(1,)](x, out, 16, 16)   # 16 threads, 0% idle
fused_softmax_constexpr[(1,)](x, out, 64, 64)   # 64 threads
```

In [ ]:
import numpy as np
import time

# Fused softmax with configurable block size
@tt.jit
def fused_softmax_constexpr(src, dst, N, BLOCK: tt.constexpr):
    tid  = tt.arange(0, BLOCK)
    mask = tid < N
    x    = tt.load(src + tid, mask=mask, other=-float('inf'))
    mx   = tt.reduce_max(x)
    e    = tt.exp(x - mx)
    s    = tt.reduce_sum(e)
    tt.store(dst + tid, e / s, mask=mask)


# --- correctness -----------------------------------------------------------
print('Correctness:')
for BLOCK in [4, 16, 27, 64]:
    N = BLOCK
    x = np.random.randn(N).astype(np.float32)
    out = np.zeros(N, dtype=np.float32)
    fused_softmax_constexpr[(1,)](x.copy(), out, N, N)
    e = np.exp(x - x.max())
    ok = np.allclose(out, e / e.sum(), atol=1e-5)
    print(f'  BLOCK={BLOCK}: {"PASS" if ok else "FAIL"}')

# --- cache separation ------------------------------------------------------
n_entries = len(fused_softmax_constexpr._cache)
print(f'\nUnique compiled kernels (one per BLOCK value): {n_entries}')

# --- backward compatibility: literal arange --------------------------------
@tt.jit
def fused_softmax_literal(src, dst, N):
    tid  = tt.arange(0, 64)
    mask = tid < N
    x    = tt.load(src + tid, mask=mask, other=-float('inf'))
    mx   = tt.reduce_max(x)
    e    = tt.exp(x - mx)
    s    = tt.reduce_sum(e)
    tt.store(dst + tid, e / s, mask=mask)

x = np.random.randn(16).astype(np.float32)
out_lit = np.zeros(16, dtype=np.float32)
fused_softmax_literal[(1,)](x.copy(), out_lit, 16)
e = np.exp(x - x.max())
ok = np.allclose(out_lit, e / e.sum(), atol=1e-5)
print(f'\nBackward compat (literal arange): {"PASS" if ok else "FAIL"}')

# --- timing ----------------------------------------------------------------
N = 64
x64 = np.random.randn(N).astype(np.float32)
REPS = 500

def time_kernel(fn, block, label):
    out = np.zeros(N, dtype=np.float32)
    fn[(1,)](x64.copy(), out, N, block)  # warmup
    t0 = time.perf_counter()
    for _ in range(REPS):
        fn[(1,)](x64.copy(), out, N, block)
    return (time.perf_counter() - t0) / REPS * 1e6

t_64 = time_kernel(fused_softmax_constexpr, 64, 'BLOCK=64')
t_16 = time_kernel(fused_softmax_constexpr, 16, 'BLOCK=16 (simulated)')
print(f'\nTiming (N=64, {REPS} reps):')
print(f'  BLOCK=64 : {t_64:.1f} µs/call')
print(f'  BLOCK=16 : {t_16:.1f} µs/call')

## 21. Shared memory — tt.sync / tt.shared_store / tt.shared_load

GPU threads within a block can communicate via fast on-chip shared memory.
Three new primitives let you load data once from global memory, store it into
shared memory, synchronize all threads, and then have every thread read from
shared memory — avoiding redundant global reads.

The tiled matvec below computes 2 output rows per block. The input vector `x`
is loaded into shared memory once and reused for both dot products.

In [ ]:
import numpy as np
import time

# --- shared memory round-trip ------------------------------------------------

@tt.jit
def shmem_roundtrip(src, dst, N, BLOCK: tt.constexpr):
    tid  = tt.arange(0, BLOCK)
    mask = tid < N
    val  = tt.load(src + tid, mask=mask)
    tt.shared_store(tid, val)
    tt.sync()
    out  = tt.shared_load(tid)
    tt.store(dst + tid, out, mask=mask)

print('Shared memory round-trip:')
for N in [4, 16]:
    x = np.random.randn(N).astype(np.float32)
    out = np.zeros(N, dtype=np.float32)
    shmem_roundtrip[(1,)](x.copy(), out, N, N)
    ok = np.allclose(out, x)
    print(f'  N={N}: {"PASS" if ok else "FAIL"}')

# --- baseline: 1-row-per-block linear (no shared memory) --------------------

@tt.jit
def baseline_linear(W_ptr, x_ptr, y_ptr, in_features, BLOCK: tt.constexpr):
    pid = tt.program_id(0)
    tid = tt.arange(0, BLOCK)
    mask = tid < in_features
    w = tt.load(W_ptr + pid * in_features + tid, mask=mask)
    x = tt.load(x_ptr + tid, mask=mask)
    dot = tt.reduce_sum(w * x)
    tt.store(y_ptr + pid, dot)

# --- tiled: 2-row-per-block linear (shared memory) --------------------------

@tt.jit
def tiled_linear(W_ptr, x_ptr, y_ptr, in_features, BLOCK: tt.constexpr):
    pid  = tt.program_id(0)
    tid  = tt.arange(0, BLOCK)
    mask = tid < in_features
    x_val = tt.load(x_ptr + tid, mask=mask)
    tt.shared_store(tid, x_val)
    tt.sync()
    x_sh = tt.shared_load(tid)
    w0   = tt.load(W_ptr + (pid * 2)     * in_features + tid, mask=mask)
    w1   = tt.load(W_ptr + (pid * 2 + 1) * in_features + tid, mask=mask)
    dot0 = tt.reduce_sum(w0 * x_sh)
    dot1 = tt.reduce_sum(w1 * x_sh)
    tt.store(y_ptr + pid * 2,     dot0)
    tt.store(y_ptr + pid * 2 + 1, dot1)

# --- correctness check ------------------------------------------------------

print('\nCorrectness:')
for out_f, in_f in [(8, 32), (16, 64)]:
    W = np.random.randn(out_f, in_f).astype(np.float32)
    x = np.random.randn(in_f).astype(np.float32)
    expected = W @ x

    y_base = np.zeros(out_f, dtype=np.float32)
    baseline_linear[(out_f,)](W.flatten().copy(), x.copy(), y_base, in_f, in_f)

    y_tile = np.zeros(out_f, dtype=np.float32)
    tiled_linear[(out_f // 2,)](W.flatten().copy(), x.copy(), y_tile, in_f, in_f)

    ok_b = np.allclose(y_base, expected, atol=1e-4)
    ok_t = np.allclose(y_tile, expected, atol=1e-4)
    match = np.allclose(y_base, y_tile, atol=1e-5)
    print(f'  {out_f}x{in_f}:  baseline={"PASS" if ok_b else "FAIL"}  '
          f'tiled={"PASS" if ok_t else "FAIL"}  match={match}')

# --- timing comparison -------------------------------------------------------

REPS = 500
out_f, in_f = 16, 64
W = np.random.randn(out_f, in_f).astype(np.float32)
x = np.random.randn(in_f).astype(np.float32)

# warmup
y = np.zeros(out_f, dtype=np.float32)
baseline_linear[(out_f,)](W.flatten().copy(), x.copy(), y, in_f, in_f)
tiled_linear[(out_f // 2,)](W.flatten().copy(), x.copy(), y, in_f, in_f)

t0 = time.perf_counter()
for _ in range(REPS):
    baseline_linear[(out_f,)](W.flatten().copy(), x.copy(), y, in_f, in_f)
t_base = (time.perf_counter() - t0) / REPS * 1e6

t0 = time.perf_counter()
for _ in range(REPS):
    tiled_linear[(out_f // 2,)](W.flatten().copy(), x.copy(), y, in_f, in_f)
t_tile = (time.perf_counter() - t0) / REPS * 1e6

print(f'\nTiming ({out_f}x{in_f} matvec, {REPS} reps):')
print(f'  Baseline ({out_f} blocks, 1 row each): {t_base:.1f} µs/call')
print(f'  Tiled    ({out_f//2} blocks, 2 rows each): {t_tile:.1f} µs/call')
print(f'  Kernel launches: baseline={out_f}, tiled={out_f//2} (2x fewer)')

## 22. Tiled GEMM — compile-time for-loop unrolling

The JIT compiler now handles `for k in range(start, stop, step)` when all
bounds are `tt.constexpr` or Python literals.  The loop body is emitted once
per iteration value — **straight-line IR, no phi nodes** — and the
accumulator chains naturally across iterations via SSA value bindings.

This enables full tiled GEMM:
```python
acc = 0.0
for k0 in range(0, K, TILE_K):   # unrolled K//TILE_K times
    acc = acc + tt.reduce_sum(a_tile * b_tile)
tt.store(C_ptr + row, acc)
```

In [15]:
import numpy as np
import time

# ── 1. loop_sum_kernel — proves for-loop accumulation ──────────────────────

@tt.jit
def loop_sum_kernel(src_ptr, dst_ptr, N: tt.constexpr, TILE: tt.constexpr):
    """Sum N elements using a tiled for loop."""
    tid = tt.arange(0, TILE)
    acc = 0.0
    for t in range(0, N, TILE):
        mask = tid < (N - t)
        x = tt.load(src_ptr + t + tid, mask=mask, other=0.0)
        acc = acc + tt.reduce_sum(x)
    tt.store(dst_ptr + tid, acc, mask=tid < 1)

print('loop_sum correctness:')
for N, TILE in [(8, 4), (12, 4), (16, 8)]:
    src = np.arange(1, N + 1, dtype=np.float32)
    dst = np.zeros(TILE, dtype=np.float32)
    loop_sum_kernel[(1,)](src, dst, N, TILE)
    ok = abs(dst[0] - src.sum()) < 1e-4
    print(f'  N={N:2d} TILE={TILE}: got {dst[0]:.1f}  expected {src.sum():.1f}  {"PASS" if ok else "FAIL"}')

# ── 2. tiled_dot_kernel — tiled matvec using for loop ──────────────────────

@tt.jit
def tiled_dot_kernel(A_ptr, B_ptr, C_ptr, K: tt.constexpr, TILE_K: tt.constexpr):
    """C[row] = dot(A[row, :], B[:]) — one block per row, tiled over K."""
    row = tt.program_id(0)
    tid = tt.arange(0, TILE_K)
    acc = 0.0
    for k0 in range(0, K, TILE_K):
        a_tile = tt.load(A_ptr + row * K + k0 + tid)
        b_tile = tt.load(B_ptr + k0 + tid)
        acc = acc + tt.reduce_sum(a_tile * b_tile)
    tt.store(C_ptr + row, acc)

print('\ntiled_dot correctness:')
for M, K, TILE_K in [(4, 8, 4), (8, 16, 4), (8, 16, 8)]:
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K).astype(np.float32)
    C = np.zeros(M, dtype=np.float32)
    tiled_dot_kernel[(M,)](A, B, C, K, TILE_K)
    max_err = float(np.abs(C - A @ B).max())
    ok = max_err < 1e-4
    print(f'  M={M} K={K:2d} TILE_K={TILE_K}: max_err={max_err:.2e}  {"PASS" if ok else "FAIL"}')

# ── 3. tiled_matmul_kernel — full GEMM with nested for loops ───────────────

@tt.jit
def tiled_matmul_kernel(A_ptr, B_ptr, C_ptr,
                         K: tt.constexpr, N: tt.constexpr,
                         TILE_K: tt.constexpr):
    """C[M,N] = A[M,K] @ B[K,N]. Nested for-loops, one block per row."""
    row = tt.program_id(0)
    tid = tt.arange(0, TILE_K)
    for col in range(N):
        acc = 0.0
        for k0 in range(0, K, TILE_K):
            a_tile = tt.load(A_ptr + row * K + k0 + tid)
            b_tile = tt.load(B_ptr + k0 * N + col + tid * N)
            acc = acc + tt.reduce_sum(a_tile * b_tile)
        tt.store(C_ptr + row * N + col, acc)

print('\ntiled_matmul correctness:')
for M, K, N, TILE_K in [(2, 4, 4, 4), (4, 8, 4, 4), (4, 8, 8, 4)]:
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    C = np.zeros((M, N), dtype=np.float32)
    tiled_matmul_kernel[(M,)](A, B, C, K, N, TILE_K)
    max_err = float(np.abs(C - A @ B).max())
    ok = max_err < 1e-4
    print(f'  M={M} K={K} N={N} TILE_K={TILE_K}: max_err={max_err:.2e}  {"PASS" if ok else "FAIL"}')

# ── timing: tiled_dot vs single-tile (no loop) ─────────────────────────────

@tt.jit
def single_tile_dot(A_ptr, B_ptr, C_ptr, K: tt.constexpr):
    """Baseline: single-tile dot product (no for loop)."""
    row = tt.program_id(0)
    tid = tt.arange(0, K)
    a = tt.load(A_ptr + row * K + tid)
    b = tt.load(B_ptr + tid)
    tt.store(C_ptr + row, tt.reduce_sum(a * b))

M, K, TILE_K, REPS = 16, 16, 4, 500
A = np.random.randn(M, K).astype(np.float32)
B = np.random.randn(K).astype(np.float32)
C = np.zeros(M, dtype=np.float32)

single_tile_dot[(M,)](A, B, C, K)                    # warmup
tiled_dot_kernel[(M,)](A, B, C, K, TILE_K)           # warmup

t0 = time.perf_counter()
for _ in range(REPS):
    single_tile_dot[(M,)](A, B, C, K)
t_single = (time.perf_counter() - t0) / REPS * 1e6

t0 = time.perf_counter()
for _ in range(REPS):
    tiled_dot_kernel[(M,)](A, B, C, K, TILE_K)
t_tiled = (time.perf_counter() - t0) / REPS * 1e6

print(f'\nTiming (M={M}, K={K}, {REPS} reps):')
print(f'  single-tile (BLOCK={K})              : {t_single:.1f} µs/call')
print(f'  tiled (TILE_K={TILE_K}, {K//TILE_K} iter): {t_tiled:.1f} µs/call')
print('  (simulator: similar — the GPU JIT-compiled version shows the real benefit)')

loop_sum correctness:
  N= 8 TILE=4: got 36.0  expected 36.0  PASS
  N=12 TILE=4: got 78.0  expected 78.0  PASS
  N=16 TILE=8: got 136.0  expected 136.0  PASS

tiled_dot correctness:
  M=4 K= 8 TILE_K=4: max_err=2.38e-07  PASS
  M=8 K=16 TILE_K=4: max_err=4.77e-07  PASS
  M=8 K=16 TILE_K=8: max_err=4.77e-07  PASS

tiled_matmul correctness:
  M=2 K=4 N=4 TILE_K=4: max_err=1.19e-07  PASS
  M=4 K=8 N=4 TILE_K=4: max_err=4.77e-07  PASS
  M=4 K=8 N=8 TILE_K=4: max_err=4.77e-07  PASS

Timing (M=16, K=16, 500 reps):
  single-tile (BLOCK=16)              : 89269.7 µs/call
  tiled (TILE_K=4, 4 iter): 112033.3 µs/call
  (simulator: similar — the GPU JIT-compiled version shows the real benefit)


## 19. Summary

In [ ]:
passed = sum(1 for _, ok in results if ok)
failed = sum(1 for _, ok in results if not ok)
total = passed + failed

print(f'\n{"="*60}')
print(f'  CPU vs GPU results:  {passed}/{total} passed, {failed} failed')
print(f'{"="*60}')

if failed:
    print('\n  Failed tests:')
    for name, ok in results:
        if not ok:
            print(f'    - {name}')

assert failed == 0, f'{failed} test(s) failed!'
print('\n  All CPU vs GPU tests passed!')